In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/jvkrishwanth/mwdataset/sub-02-20260605T131514Z-3-001/sub-02/sub-02_sessions.tsv
/kaggle/input/datasets/jvkrishwanth/mwdataset/sub-02-20260605T131514Z-3-001/sub-02/sub-02_events.tsv
/kaggle/input/datasets/jvkrishwanth/mwdataset/sub-02-20260605T131514Z-3-001/sub-02/eeg/sub-02_ses-7_task-BreathCounting_eeg.json
/kaggle/input/datasets/jvkrishwanth/mwdataset/sub-02-20260605T131514Z-3-001/sub-02/eeg/sub-02_ses-8_task-BreathCounting_channels.tsv
/kaggle/input/datasets/jvkrishwanth/mwdataset/sub-02-20260605T131514Z-3-001/sub-02/eeg/sub-02_ses-5_task-BreathCounting_electrodes.tsv
/kaggle/input/datasets/jvkrishwanth/mwdataset/sub-02-20260605T131514Z-3-001/sub-02/eeg/sub-02_ses-10_task-BreathCounting_channels.tsv
/kaggle/input/datasets/jvkrishwanth/mwdataset/sub-02-20260605T131514Z-3-001/sub-02/eeg/sub-02_ses-8_task-BreathCounting_eeg.json
/kaggle/input/datasets/jvkrishwanth/mwdataset/sub-02-20260605T131514Z-3-001/sub-02/eeg/sub-02_ses-4_task-BreathCounting_eeg.json
/kaggle

In [2]:
"""Kaggle-ready DeepMLP comparison of raw and ICA-cleaned EEG.

Inputs are the BDF recordings and their channel TSV files; no feature CSV is
required.  The two conditions are created from the same filtered recordings:
Raw has no EXG regression and ICA-cleaned has the linear EXG contribution
regressed from each EEG channel.  This is not an ICA-cleaning pipeline.
"""

import os

os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

from pathlib import Path
import gc
import random
import warnings

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.integrate import trapezoid
from scipy.signal import butter, hilbert, sosfiltfilt, welch
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


# =============================================================================
# Kaggle paths and fixed analysis configuration
# =============================================================================

DATASET_ROOT = Path("/kaggle/input/datasets/jvkrishwanth/mwdataset")
OUTDIR = Path("/kaggle/working/dataset2_deepmlp_raw_vs_ica_cleaned")
PLOT_DIR = OUTDIR / "plots"

SUBJECTS = ["sub-01", "sub-02"]
# Matches the earlier DeepMLP notebook: use every available session (1–11).
SESSIONS = range(1, 12)
TARGET_SFREQ = 256.0
RANDOM_SEED = 42

# The BDF recordings and their metadata are stored in different archive folders.
BDF_ARCHIVES = {
    "sub-01": {
        1: "sub-01-20260605T131512Z-3-002", 2: "sub-01-20260605T131512Z-3-003",
        3: "sub-01-20260605T131512Z-3-002", 4: "sub-01-20260605T131512Z-3-002",
        5: "sub-01-20260605T131512Z-3-002", 6: "sub-01-20260605T131512Z-3-001",
        7: "sub-01-20260605T131512Z-3-001", 8: "sub-01-20260605T131512Z-3-001",
        9: "sub-01-20260605T131512Z-3-001", 10: "sub-01-20260605T131512Z-3-003",
        11: "sub-01-20260605T131512Z-3-003",
    },
    "sub-02": {
        1: "sub-02-20260605T131514Z-3-003", 2: "sub-02-20260605T131514Z-3-002",
        3: "sub-02-20260605T131514Z-3-002", 4: "sub-02-20260605T131514Z-3-001",
        5: "sub-02-20260605T131514Z-3-001", 6: "sub-02-20260605T131514Z-3-002",
        7: "sub-02-20260605T131514Z-3-002", 8: "sub-02-20260605T131514Z-3-001",
        9: "sub-02-20260605T131514Z-3-001", 10: "sub-02-20260605T131514Z-3-001",
        11: "sub-02-20260605T131514Z-3-003",
    },
}
METADATA_ARCHIVES = {
    "sub-01": "sub-01-20260605T131512Z-3-001",
    "sub-02": "sub-02-20260605T131514Z-3-001",
}

EVENT_TRIAL_START = 10
EVENT_MW_REPORT = 30
EVENT_START_COUNTING = 50
EPOCH_DURATION = 10.0
MW_START_OFFSET = -11.0
FOCUS_START_OFFSET = 1.0

WINDOW_LENGTH = 1.5
WINDOW_STEP = 0.5
BANDS = {
    "Delta": (1.0, 4.0),
    "Theta": (4.0, 8.0),
    "Alpha": (8.0, 12.0),
    "Beta": (13.0, 30.0),
    "Gamma": (30.0, 45.0),
}

EPOCHS = 150
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 25

CONDITIONS = ["Raw", "ICA-cleaned"]
METRIC_ORDER = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]
CONDITION_COLORS = {"Raw": "#66c2a5", "ICA-cleaned": "#fc8d62"}


# =============================================================================
# Reproducibility and plot formatting
# =============================================================================

def set_seed(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


warnings.filterwarnings("ignore")
mne.set_log_level("WARNING")
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({
    "figure.dpi": 100,
    "savefig.dpi": 100,
    "axes.grid": True,
    "grid.color": "#c7c7c7",
    "grid.linewidth": 1.0,
    "axes.edgecolor": "#c7c7c7",
    "axes.linewidth": 1.0,
})


# =============================================================================
# Direct BDF loading and matched raw/ICA-cleaned preprocessing
# =============================================================================

def session_paths(subject, session):
    session_label = f"ses-{session}"
    eeg_dir = DATASET_ROOT / BDF_ARCHIVES[subject][session] / subject / "eeg"
    metadata_dir = DATASET_ROOT / METADATA_ARCHIVES[subject] / subject / "eeg"
    return (
        eeg_dir / f"{subject}_{session_label}_task-BreathCounting_eeg.bdf",
        metadata_dir / f"{subject}_{session_label}_task-BreathCounting_channels.tsv",
    )


def deduplicate_events(events):
    if len(events) == 0:
        return events
    events_df = pd.DataFrame(events, columns=["sample", "previous", "code"])
    events_df = events_df.drop_duplicates(["sample", "code"], keep="first")
    return events_df.sort_values(["sample", "code"])[["sample", "previous", "code"]].to_numpy(dtype=int)


def get_channel_names(raw, channels_tsv):
    channels = pd.read_csv(channels_tsv, sep="\t")
    eeg_names = channels.loc[
        channels["channelTypes"].fillna("").str.upper().eq("EEG"), "name"
    ].astype(str).tolist()
    eeg_names = [channel for channel in eeg_names if channel in raw.ch_names]
    exg_names = [f"EXG{number}" for number in range(1, 9) if f"EXG{number}" in raw.ch_names]
    if not eeg_names:
        raise RuntimeError("No channel labelled EEG in the TSV is present in the BDF recording.")
    return eeg_names, exg_names


def ica_clean(raw, eeg_names, exg_names, correlation_threshold=0.30):
    """Remove ICA components correlated with EXG channels without dropping trials."""
    cleaned = raw.copy()
    if not exg_names:
        return cleaned.pick(eeg_names)
    ica = mne.preprocessing.ICA(
        n_components=0.99, method="fastica", random_state=42, max_iter="auto",
    )
    ica.fit(cleaned, picks=eeg_names, verbose=False)
    scores = [
        np.asarray(ica.score_sources(cleaned, target=cleaned.get_data(picks=[name])[0]), dtype=float)
        for name in exg_names
    ]
    excluded = np.flatnonzero(np.max(np.abs(np.vstack(scores)), axis=0) >= correlation_threshold)
    ica.apply(cleaned, exclude=excluded, verbose=False)
    return cleaned.pick(eeg_names)

def build_epoch_events(events, sfreq):
    rows = []
    for sample, _, code in events:
        if code == EVENT_MW_REPORT:
            start, label = sample + round(MW_START_OFFSET * sfreq), 1
        elif code in (EVENT_TRIAL_START, EVENT_START_COUNTING):
            start, label = sample + round(FOCUS_START_OFFSET * sfreq), 0
        else:
            continue
        if start >= 0:
            rows.append([start, 0, label])
    return deduplicate_events(np.asarray(rows, dtype=int)) if rows else np.empty((0, 3), dtype=int)


def load_conditions(subject, session):
    bdf_path, channels_tsv = session_paths(subject, session)
    if not bdf_path.is_file() or not channels_tsv.is_file():
        raise FileNotFoundError(f"Input missing for {subject}, session {session}.")
    raw = mne.io.read_raw_bdf(bdf_path, preload=True, stim_channel="Status", verbose=False)
    events = mne.find_events(raw, stim_channel="Status", shortest_event=1, consecutive=True, verbose=False)
    eeg_names, exg_names = get_channel_names(raw, channels_tsv)
    original_sfreq = raw.info["sfreq"]
    raw.pick(eeg_names + exg_names)
    raw.resample(TARGET_SFREQ, npad="auto", verbose=False)
    events[:, 0] = np.rint(events[:, 0] * TARGET_SFREQ / original_sfreq).astype(int)
    events = deduplicate_events(events)
    raw.filter(0.5, 45.0, fir_design="firwin", verbose=False)

    # Raw is filtered EEG only: no EXG regression is applied.
    raw_condition = raw.copy().pick(eeg_names)
    # ICA-cleaned starts from the same filtered recording and removes EXG-correlated ICA components.
    clean_condition = ica_clean(raw, eeg_names, exg_names)
    return {"Raw": raw_condition, "ICA-cleaned": clean_condition}, events


def make_epochs(raw, events):
    epoch_events = build_epoch_events(events, raw.info["sfreq"])
    if not len(epoch_events):
        raise RuntimeError("No valid MW or Focus events are available for epoching.")
    return mne.Epochs(
        raw, epoch_events, event_id={"Focus": 0, "MW": 1}, tmin=0.0,
        tmax=EPOCH_DURATION, baseline=None, reject=None, flat=None,
        preload=True, event_repeated="drop", verbose=False,
    )


# =============================================================================
# Direct EEG feature extraction: one feature vector per overlapping window
# =============================================================================

def bandpass_filter(data, low, high, sfreq):
    sos = butter(4, [low / (sfreq / 2), high / (sfreq / 2)], btype="band", output="sos")
    return sosfiltfilt(sos, data, axis=-1)


def count_bursts(envelope):
    threshold = envelope.mean() + 2.0 * envelope.std()
    return float(np.sum(np.diff((envelope > threshold).astype(int)) == 1))


def plv_matrix(data, sfreq, low, high):
    filtered = bandpass_filter(data, low, high, sfreq)
    phase_exp = np.exp(1j * np.angle(hilbert(filtered, axis=-1)))
    return np.abs(phase_exp @ phase_exp.conj().T / data.shape[-1])


def extract_window_vectors(epochs, subject, session, condition):
    """Mirror the older notebook's PSD, envelope/burst, and PLV feature set."""
    rows = []
    window_samples = round(WINDOW_LENGTH * epochs.info["sfreq"])
    step_samples = round(WINDOW_STEP * epochs.info["sfreq"])
    for epoch_index, (epoch, event) in enumerate(zip(epochs.get_data(), epochs.events), start=1):
        label = int(event[2] == 1)
        for window_index, start in enumerate(range(0, epoch.shape[-1] - window_samples + 1, step_samples)):
            data = epoch[:, start:start + window_samples]
            frequencies, psd = welch(data, epochs.info["sfreq"], nperseg=min(256, window_samples), axis=-1)
            masks = {
                band: (frequencies >= low) & ((frequencies <= high) if band == "Gamma" else (frequencies < high))
                for band, (low, high) in BANDS.items()
            }
            total_mask = (frequencies >= 1.0) & (frequencies <= 45.0)
            total_power = psd[:, total_mask].mean(axis=-1)
            values = []
            alpha_envelope = np.abs(hilbert(bandpass_filter(data, 8.0, 12.0, epochs.info["sfreq"]), axis=-1))
            theta_envelope = np.abs(hilbert(bandpass_filter(data, 4.0, 8.0, epochs.info["sfreq"]), axis=-1))
            beta_envelope = np.abs(hilbert(bandpass_filter(data, 13.0, 30.0, epochs.info["sfreq"]), axis=-1))
            # Per channel: five absolute/relative powers plus alpha/theta/beta envelopes.
            for channel_index in range(data.shape[0]):
                for band in BANDS:
                    band_power = psd[channel_index, masks[band]].mean()
                    values.extend([band_power, band_power / (total_power[channel_index] + 1e-8)])
                for envelope in (alpha_envelope, theta_envelope, beta_envelope):
                    signal = envelope[channel_index]
                    mean = signal.mean()
                    values.extend([mean, signal.var(), signal.std() / (mean + 1e-8), count_bursts(signal)])
            # Pairwise alpha, theta, and beta PLV values, as in the prior notebook.
            for matrix in (
                plv_matrix(data, epochs.info["sfreq"], 8.0, 12.0),
                plv_matrix(data, epochs.info["sfreq"], 4.0, 8.0),
                plv_matrix(data, epochs.info["sfreq"], 13.0, 30.0),
            ):
                values.extend(matrix[np.triu_indices(data.shape[0], k=1)])
            rows.append({
                "Condition": condition,
                "Subject": subject,
                "Session": session,
                "Epoch_ID": f"{subject}_ses-{session}_epoch-{epoch_index}",
                "Window": window_index,
                "Label": label,
                "Features": np.asarray(values, dtype=np.float32),
            })
    return rows


# =============================================================================
# DeepMLP and group-preserving out-of-fold evaluation
# =============================================================================

class DeepMLP(nn.Module):
    def __init__(self, input_dimension):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dimension, 1024), nn.BatchNorm1d(1024), nn.GELU(), nn.Dropout(0.50),
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.GELU(), nn.Dropout(0.40),
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.GELU(), nn.Dropout(0.30),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.GELU(), nn.Dropout(0.20),
            nn.Linear(128, 1),
        )

    def forward(self, features):
        return self.layers(features).squeeze(-1)


def evaluate_loader(model, loader, criterion, device):
    model.eval()
    all_labels, all_probabilities, total_loss = [], [], 0.0
    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            logits = model(batch_x)
            total_loss += criterion(logits, batch_y).item() * len(batch_x)
            all_labels.extend(batch_y.cpu().numpy())
            all_probabilities.extend(torch.sigmoid(logits).cpu().numpy())
    return np.asarray(all_labels, dtype=int), np.asarray(all_probabilities), total_loss / len(loader.dataset)


def train_and_predict(X_train, y_train, X_test, y_test, device):
    """Replicate the older MLP training/early-stopping procedure for one fold."""
    train_data = TensorDataset(torch.tensor(X_train), torch.tensor(y_train, dtype=torch.float32))
    test_data = TensorDataset(torch.tensor(X_test), torch.tensor(y_test, dtype=torch.float32))
    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)
    model = DeepMLP(X_train.shape[1]).to(device)
    positive_count = max(1, int(y_train.sum()))
    negative_count = len(y_train) - positive_count
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(negative_count / positive_count, device=device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, "min", patience=10, factor=0.5)

    best_loss, remaining_patience, best_state = np.inf, PATIENCE, None
    for _ in range(EPOCHS):
        model.train()
        running_loss = 0.0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(batch_x), batch_y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(batch_x)
        _, _, test_loss = evaluate_loader(model, test_loader, criterion, device)
        scheduler.step(test_loss)
        if test_loss < best_loss:
            best_loss, remaining_patience = test_loss, PATIENCE
            best_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
        else:
            remaining_patience -= 1
            if remaining_patience == 0:
                break

    model.load_state_dict(best_state)
    _, probabilities, _ = evaluate_loader(model, test_loader, criterion, device)
    return probabilities


def evaluate_condition(condition_table, device):
    """Match the older protocol: stratified epoch folds separately per subject."""
    feature_matrix = np.vstack(condition_table["Features"].to_numpy())
    labels = condition_table["Label"].to_numpy(dtype=int)
    all_true, all_predictions, all_probabilities = [], [], []

    for subject in sorted(condition_table["Subject"].unique()):
        subject_index = np.flatnonzero(condition_table["Subject"].to_numpy() == subject)
        subject_epochs = condition_table.iloc[subject_index]["Epoch_ID"].to_numpy()
        unique_epochs, first_epoch_rows = np.unique(subject_epochs, return_index=True)
        epoch_labels = labels[subject_index[first_epoch_rows]]
        number_of_splits = min(5, np.bincount(epoch_labels).min())
        if number_of_splits < 2:
            print(f"  Skipping {subject}: fewer than two epochs in one class.")
            continue
        splitter = StratifiedKFold(n_splits=number_of_splits, shuffle=True, random_state=RANDOM_SEED)
        for fold, (train_epoch_index, test_epoch_index) in enumerate(splitter.split(unique_epochs, epoch_labels), start=1):
            print(f"  {subject}, fold {fold}")
            train_epochs = unique_epochs[train_epoch_index]
            test_epochs = unique_epochs[test_epoch_index]
            train_index = subject_index[np.isin(subject_epochs, train_epochs)]
            test_index = subject_index[np.isin(subject_epochs, test_epochs)]
            imputer = SimpleImputer(strategy="median")
            scaler = StandardScaler()
            X_train = scaler.fit_transform(imputer.fit_transform(feature_matrix[train_index])).astype(np.float32)
            X_test = scaler.transform(imputer.transform(feature_matrix[test_index])).astype(np.float32)
            set_seed(RANDOM_SEED + fold)
            window_probabilities = train_and_predict(
                X_train, labels[train_index], X_test, labels[test_index], device,
            )
            # The older notebook scores one prediction per epoch by averaging its
            # overlapping-window probabilities rather than scoring each window.
            test_epoch_ids = subject_epochs[np.isin(subject_epochs, test_epochs)]
            for epoch_id in test_epochs:
                epoch_mask = test_epoch_ids == epoch_id
                epoch_probability = window_probabilities[epoch_mask].mean()
                all_true.append(labels[test_index][epoch_mask][0])
                all_probabilities.append(epoch_probability)
                all_predictions.append(int(epoch_probability >= 0.5))

    true_labels = np.asarray(all_true, dtype=int)
    probabilities = np.asarray(all_probabilities)
    predictions = np.asarray(all_predictions, dtype=int)
    if not len(true_labels):
        raise RuntimeError("No out-of-fold epoch predictions were generated.")
    metrics = {
        "Accuracy": accuracy_score(true_labels, predictions),
        "Precision": precision_score(true_labels, predictions, zero_division=1),
        "Recall": recall_score(true_labels, predictions, zero_division=1),
        "F1": f1_score(true_labels, predictions, zero_division=1),
        "ROC-AUC": roc_auc_score(true_labels, probabilities),
    }
    return true_labels, predictions, probabilities, metrics


# =============================================================================
# Figures matched to the requested format, scale, and colors
# =============================================================================

def save_confusion_matrix(y_true, y_pred, condition, common_vmax):
    matrix = confusion_matrix(y_true, y_pred, labels=[0, 1])
    figure, axis = plt.subplots(figsize=(10, 8))
    sns.heatmap(
        matrix, annot=True, fmt="d", cmap="Blues", cbar=False, square=True,
        vmin=0, vmax=common_vmax, xticklabels=["Focused", "MW"],
        yticklabels=["Focused", "MW"], ax=axis,
    )
    axis.set_title(f"{condition}: out-of-fold confusion matrix", pad=12)
    axis.set_xlabel("Predicted")
    axis.set_ylabel("True")
    figure.tight_layout()
    filename = f"{condition.lower().replace('-', '_')}_confusion_matrix.png"
    figure.savefig(PLOT_DIR / filename, bbox_inches="tight")
    plt.close(figure)


def save_metric_comparison(metric_rows):
    metrics = pd.DataFrame(metric_rows)
    figure, axis = plt.subplots(figsize=(20, 12))
    positions = np.arange(len(METRIC_ORDER))
    width = 0.4
    for offset, condition in [(-width / 2, "Raw"), (width / 2, "ICA-cleaned")]:
        values = metrics.loc[metrics["Condition"].eq(condition), "Score"].to_numpy()
        bars = axis.bar(positions + offset, values, width, color=CONDITION_COLORS[condition], label=condition)
        axis.bar_label(bars, labels=[f"{value:.3f}" for value in values], padding=3, fontsize=12)
    axis.set_title("DeepMLP: raw vs ICA-cleaned EEG classification", pad=12)
    axis.set_xlabel("Metric")
    axis.set_ylabel("Score")
    axis.set_xticks(positions, METRIC_ORDER)
    axis.set_ylim(0, 1.0)
    axis.legend(title="Condition", loc="upper right")
    figure.tight_layout()
    figure.savefig(PLOT_DIR / "raw_vs_ica_cleaned_classification_metrics.png", bbox_inches="tight")
    plt.close(figure)


# =============================================================================
# Execution
# =============================================================================

def main():
    if not DATASET_ROOT.is_dir():
        raise FileNotFoundError(f"Kaggle dataset directory not found: {DATASET_ROOT}")
    set_seed()
    OUTDIR.mkdir(parents=True, exist_ok=True)
    PLOT_DIR.mkdir(parents=True, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    all_rows, failures = [], []
    for subject in SUBJECTS:
        for session in SESSIONS:
            try:
                print(f"Extracting BDF features: {subject}, session {session}")
                conditions, events = load_conditions(subject, session)
                for condition, raw in conditions.items():
                    epochs = make_epochs(raw, events)
                    all_rows.extend(extract_window_vectors(epochs, subject, session, condition))
                    del epochs
                del conditions, events
                gc.collect()
            except Exception as error:
                print(f"FAILED {subject}, session {session}: {error}")
                failures.append({"Subject": subject, "Session": session, "Error": str(error)})

    if not all_rows:
        raise RuntimeError("No BDF recordings were processed successfully.")
    feature_table = pd.DataFrame(all_rows)
    feature_table.drop(columns="Features").to_csv(OUTDIR / "window_sample_metadata.csv", index=False)
    pd.DataFrame(failures).to_csv(OUTDIR / "failed_runs.csv", index=False)

    results, metric_rows = {}, []
    for condition in CONDITIONS:
        print(f"\nTraining DeepMLP on {condition}")
        condition_table = feature_table[feature_table["Condition"].eq(condition)].reset_index(drop=True)
        y_true, y_pred, y_probability, metrics = evaluate_condition(condition_table, device)
        results[condition] = {"y_true": y_true, "y_pred": y_pred, "y_probability": y_probability}
        metric_rows.extend({"Condition": condition, "Metric": metric, "Score": value} for metric, value in metrics.items())

    metrics_df = pd.DataFrame(metric_rows)
    metrics_df["Metric"] = pd.Categorical(metrics_df["Metric"], METRIC_ORDER, ordered=True)
    metrics_df = metrics_df.sort_values(["Metric", "Condition"])
    metrics_df.to_csv(OUTDIR / "raw_vs_ica_cleaned_deepmlp_metrics.csv", index=False)

    shared_vmax = max(confusion_matrix(result["y_true"], result["y_pred"], labels=[0, 1]).max() for result in results.values())
    for condition in CONDITIONS:
        save_confusion_matrix(results[condition]["y_true"], results[condition]["y_pred"], condition, shared_vmax)
    save_metric_comparison(metric_rows)

    print("\nOut-of-fold DeepMLP metrics:")
    print(metrics_df.pivot(index="Metric", columns="Condition", values="Score"))
    print(f"\nPlots: {PLOT_DIR}")
    print(f"Tables: {OUTDIR}")


if __name__ == "__main__":
    main()


Using device: cpu
Extracting BDF features: sub-01, session 1
Extracting BDF features: sub-01, session 2
Extracting BDF features: sub-01, session 3
Extracting BDF features: sub-01, session 4
Extracting BDF features: sub-01, session 5
Extracting BDF features: sub-01, session 6
Extracting BDF features: sub-01, session 7
Extracting BDF features: sub-01, session 8
Extracting BDF features: sub-01, session 9
Extracting BDF features: sub-01, session 10
Extracting BDF features: sub-01, session 11
Extracting BDF features: sub-02, session 1
Extracting BDF features: sub-02, session 2
Extracting BDF features: sub-02, session 3
Extracting BDF features: sub-02, session 4
Extracting BDF features: sub-02, session 5
Extracting BDF features: sub-02, session 6
Extracting BDF features: sub-02, session 7
Extracting BDF features: sub-02, session 8
Extracting BDF features: sub-02, session 9
Extracting BDF features: sub-02, session 10
Extracting BDF features: sub-02, session 11

Training DeepMLP on Raw
  sub-0